# CSE425: Exploratory Data Analysis (EDA)
## Free Music Archive (FMA-Small) Analysis & Graph Construction

In [ ]:
import os
import sys
import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

project_root = os.path.abspath(".." if os.path.exists("../src") else ".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils import load_config, resolve_paths
from src.datasets import FMAMetadata

sns.set_theme(style="whitegrid", font_scale=1.1)
config = load_config("config.yaml")
resolve_paths(config, project_root)

In [ ]:
# 1. Load FMA Metadata
metadata_dir = config["dataset"]["metadata_dir"]
print(f"Loading metadata from: {metadata_dir}")

meta = FMAMetadata(metadata_dir, subset="small")
print(f"Total FMA-small tracks: {len(meta.track_ids)}")
print(f"Target Genre Classes ({len(meta.genre_labels)}): {meta.genre_labels}")

In [ ]:
# 2. Analyze Predefined Splits (Artist-Leak-Free)
splits = meta.get_split_track_ids()
split_counts = {k: len(v) for k, v in splits.items()}

plt.figure(figsize=(6, 4))
sns.barplot(x=list(split_counts.keys()), y=list(split_counts.values()), palette="Blues_d")
plt.title("FMA-Small Train / Val / Test Track Distribution")
plt.ylabel("Number of Tracks")
for i, v in enumerate(split_counts.values()):
    plt.text(i, v + 50, f"{v} ({v/len(meta.track_ids)*100:.1f}%)", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
# 3. Multi-Label Label Distribution Analysis
all_labels = np.array([meta.get_multi_hot_label(tid).numpy() for tid in meta.track_ids])
label_sums = all_labels.sum(axis=0)

plt.figure(figsize=(9, 4))
sns.barplot(x=meta.genre_labels, y=label_sums, palette="viridis")
plt.title("Genre Label Frequency (Multi-Label)")
plt.ylabel("Positive Occurrences")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 4. Sample Audio & Graph Feature Visualisation
from src.audio_features import extract_audio_features
from src.graph_builder import build_graph_from_features

sample_tid = meta.track_ids[0]
prefix = f"{sample_tid:06d}"[:3]
mp3_path = os.path.join(config["dataset"]["raw_audio_dir"], prefix, f"{sample_tid:06d}.mp3")

if os.path.exists(mp3_path):
    segments = extract_audio_features(mp3_path, sr=22050, segment_length_s=5)
    graph = build_graph_from_features(segments, similarity_threshold=0.7)
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    im1 = axs[0].imshow(segments[0]["log_mel"], aspect="auto", origin="lower", cmap="magma")
    axs[0].set_title("Segment 1: Log-Mel Spectrogram")
    axs[0].set_ylabel("Mel Bins (128)")
    axs[0].set_xlabel("Time Frames")
    plt.colorbar(im1, ax=axs[0])
    
    im2 = axs[1].imshow(segments[0]["chroma"], aspect="auto", origin="lower", cmap="coolwarm")
    axs[1].set_title("Segment 1: Chroma Features")
    axs[1].set_ylabel("Chroma Bins (12)")
    axs[1].set_xlabel("Time Frames")
    plt.colorbar(im2, ax=axs[1])
    
    plt.tight_layout()
    plt.show()
    print(f"Built graph for track {sample_tid}: {graph.num_nodes} nodes, {graph.edge_index.size(1)} edges.")
else:
    print(f"Audio file {mp3_path} not found locally — skipping audio feature plot.")